In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

df_product = spark.table("olist.bronze.olist_product")

# function to remove double quotes in fileds
def clean_column(df):
    new_columns = [
        col.replace('"', '') for col in df.columns
    ]
    return df.toDF(*new_columns)

df_product = clean_column(df_product)
  

In [0]:
# renaming the column names
df_product = (df_product
    .withColumnRenamed("product_name_lenght","product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
)

In [0]:
cols = [
    "product_id",
    "product_category_name",
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]
# Step 1: Replace empty strings with null
for col in cols:
    df_product = df_product.withColumn(
        col,
        F.when(F.col(col) == "", None).otherwise(F.col(col))
    )



In [0]:
int_cols = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]
for col in int_cols:
    df_product = df_product.withColumn(
        col,
        F.col(col).cast(IntegerType())
    )
df_product = df_product.fillna({col: 0 for col in int_cols})
df_product = df_product.fillna({"product_category_name": "unknown"})

In [0]:

# saving as delta tables
df_product.write.format("delta").mode("overwrite").saveAsTable("olist.silver.olist_product")